In [10]:
# ==============================================================================
# Cell 1: Setup, Data Loading, and Helper Functions
# ==============================================================================
import json
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import calmap
import warnings
from collections import defaultdict
import io
import base64
import panel as pn

# Initialize Panel and suppress warnings
pn.extension() 
warnings.filterwarnings('ignore', category=UserWarning, module='calmap')

# Load the JSON data
json_file_path = r"C:\Github Code\kobostatisticsdatareader\Output\kobo_consolidated_20260423_223457.json"
data = None
try:
    with open(json_file_path, 'r',encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Data loaded successfully from {json_file_path}.")
except FileNotFoundError:
    print(f"❌ Error: '{json_file_path}' not found.")

def format_duration(minutes):
    """Formats minutes into a human-readable string."""
    hours, mins = divmod(int(minutes), 60)
    return f"{hours}h {mins}m"

# ==============================================================================
# Cell 2: Data Preprocessing
# ==============================================================================
processed_data = {
    'daily_read_time': defaultdict(float),
    'time_of_day_durations': defaultdict(float),
    'finished_books': [],
    'currently_reading': [],
    'monthly_stats': defaultdict(lambda: {'duration': 0, 'sessions': 0}),
    'total_duration': 0
}

if data and 'reading_dashboard' in data:
    dashboard_data = data['reading_dashboard']
    
    for filepath, item in dashboard_data.items():
        meta = item.get('metadata', {})
        metrics = item.get('metrics', {})
        sessions = item.get('sessions', [])
        
        title = meta.get('title', 'Unknown Title')
        author = meta.get('author', 'Unknown Author')
        percent_complete = meta.get('percent_complete', 0)
        avg_wpm = metrics.get('avg_wpm', 0)
        
        # Categorize books
        book_display = f"**{title}** by {author} (Avg WPM: {avg_wpm:.1f})"
        if percent_complete >= 100:
            processed_data['finished_books'].append(book_display)
        elif percent_complete > 0 or len(sessions) > 0:
            processed_data['currently_reading'].append(f"{book_display} - {percent_complete}% Complete")

        # Process Sessions
        for session in sessions:
            try:
                # Handle standard ISO format
                dt = datetime.fromisoformat(session['timestamp'])
            except ValueError:
                continue
                
            dur_mins = session.get('duration_minutes', 0)
            processed_data['total_duration'] += dur_mins
            
            # Daily read time (for calendar/streaks)
            processed_data['daily_read_time'][dt.date()] += dur_mins
            
            # Time of day
            hour = dt.hour
            if 5 <= hour < 12: period = 'Morning (5am-12pm)'
            elif 12 <= hour < 17: period = 'Afternoon (12pm-5pm)'
            elif 17 <= hour < 21: period = 'Evening (5pm-9pm)'
            else: period = 'Night (9pm-5am)'
            processed_data['time_of_day_durations'][period] += dur_mins
            
            # Monthly aggregations
            month_key = dt.strftime('%Y-%m')
            processed_data['monthly_stats'][month_key]['duration'] += dur_mins
            processed_data['monthly_stats'][month_key]['sessions'] += 1

# Calculate Streaks
sorted_dates = sorted(processed_data['daily_read_time'].keys())
longest_streak = current_streak = 0
if sorted_dates:
    longest_streak = current_streak = 1
    for i in range(1, len(sorted_dates)):
        if sorted_dates[i] == sorted_dates[i-1] + timedelta(days=1):
            current_streak += 1
        else:
            longest_streak = max(longest_streak, current_streak)
            current_streak = 1
    longest_streak = max(longest_streak, current_streak)
    
    # Check if streak is active
    if (datetime.now().date() - sorted_dates[-1]).days > 1:
        current_streak = 0

# ==============================================================================
# Cell 3: Define Dashboard Sections
# ==============================================================================
report_pane = pn.Column("### Overview Report")
calendar_pane = pn.Column("### Streaks & Calendar")
time_pane = pn.Column("### Time of Day")
books_pane = pn.Column("### Library Status")

if data:
    # 1. Report Pane
    report_pane.append(pn.pane.Markdown(f"**Total Reading Time:** {format_duration(processed_data['total_duration'])}"))
    for month, stats in sorted(processed_data['monthly_stats'].items(), reverse=True):
        report_pane.append(pn.pane.Markdown(f"- **{month}**: {format_duration(stats['duration'])} across {stats['sessions']} sessions"))

    # 2. Calendar Pane
    calendar_pane.append(pn.pane.Markdown(f"🔥 **Longest Streak:** {longest_streak} days | 🚀 **Current Streak:** {current_streak} days"))
    
    if processed_data['daily_read_time']:
        all_days = pd.Series(processed_data['daily_read_time'])
        all_days.index = pd.to_datetime(all_days.index)
        
        buffer = io.BytesIO()
        fig, ax = plt.subplots(figsize=(16, 8))
        calmap.yearplot(all_days, year=datetime.now().year, cmap='Blues', ax=ax)
        ax.set_title(f'Reading Heatmap (Minutes Read)', fontsize=16)
        fig.savefig(buffer, format='png', bbox_inches='tight')
        plt.close(fig)
        buffer.seek(0)
        img_str = base64.b64encode(buffer.read()).decode('utf-8')
        calendar_pane.append(pn.pane.HTML(f'<img src="data:image/png;base64,{img_str}" style="width:100%; height:auto;">'))

    # 3. Time of Day Pane
    ordered_periods = ['Morning (5am-12pm)', 'Afternoon (12pm-5pm)', 'Evening (5pm-9pm)', 'Night (9pm-5am)']
    sorted_durs = {p: processed_data['time_of_day_durations'].get(p, 0) for p in ordered_periods}
    
    if any(sorted_durs.values()):
        buffer = io.BytesIO()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(sorted_durs.keys(), [v/60 for v in sorted_durs.values()], color=['skyblue', 'orange', 'salmon', 'darkblue'])
        ax.set_title('Reading Time by Time of Day', fontsize=14)
        ax.set_ylabel('Hours Read')
        plt.xticks(rotation=15)
        fig.savefig(buffer, format='png', bbox_inches='tight')
        plt.close(fig)
        buffer.seek(0)
        img_str = base64.b64encode(buffer.read()).decode('utf-8')
        time_pane.append(pn.pane.HTML(f'<img src="data:image/png;base64,{img_str}" style="width:100%; max-width:600px;">'))

    # 4. Books Pane
    books_pane.append(pn.pane.Markdown("#### Currently Reading"))
    books_pane.append(pn.pane.Markdown("\n".join([f"- {b}" for b in processed_data['currently_reading']]) if processed_data['currently_reading'] else "Nothing currently in progress."))
    
    books_pane.append(pn.pane.Markdown("#### Finished Books"))
    books_pane.append(pn.pane.Markdown("\n".join([f"- {b}" for b in processed_data['finished_books']]) if processed_data['finished_books'] else "No books finished yet."))

# ==============================================================================
# Cell 4: Create Panel Tabs and Save
# ==============================================================================
dashboard = pn.Tabs(
    ('Monthly Report', report_pane),
    ('Streaks & Calendar', calendar_pane),
    ('Time of Day', time_pane),
    ('Library Status', books_pane)
)

output_html_file = "kobo_dashboard.html"
dashboard.save(output_html_file, embed=True)
print(f"✅ Dashboard saved successfully to {output_html_file}.")

✅ Data loaded successfully from C:\Github Code\kobostatisticsdatareader\Output\kobo_consolidated_20260423_223457.json.
✅ Dashboard saved successfully to kobo_dashboard.html.
